# Giao diện người dùng tạo sinh mở

Trong bài học này, chúng ta sẽ tiến tới cấp độ linh hoạt nhất của GenUI. Thay vì bị giới hạn bởi các component đã được đăng ký trước hay các schema cố định, AI agent sẽ có khả năng định tuyến đến các ứng dụng hoàn chỉnh - ví dụ như Excalidraw - thông qua **MCP app** (đây cũng chính là giao thức ứng dụng được sử dụng bởi Claude, ChatGPT và các nền tảng AI khác).

## 📋 Mục tiêu học tập

1. **Kích hoạt GenUI mở** - Cho phép agent tạo ra giao diện người dùng bất kỳ một cách tự động.
2. **Hiểu về MCP app** - Giao thức cho phép các MCP server cung cấp các ứng dụng tương tác tới các nền tảng AI.
3. **Kết nối một MCP app** - Tích hợp bảng vẽ Excalidraw vào ứng dụng CopilotKit của bạn.

---

## 🛠 Chuẩn bị môi trường

Trước khi bắt đầu, chúng ta cần thiết lập môi trường, cài đặt các thư viện cần thiết và tải các API key.

In [1]:
# Cài đặt các thư viện frontend (Node.js)
from helper import install_frontend
install_frontend()

# Tải API key từ file .env
from helper import load_api_keys
load_api_keys()

Installing frontend dependencies ...

up to date, audited 949 packages in 3s

248 packages are looking for funding
  run `npm fund` for details

21 vulnerabilities (2 low, 17 moderate, 2 high)

To address issues that do not require attention, run:
  npm audit fix

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
npm warn allow-scripts 3 packages have install scripts not yet covered by allowScripts:
npm warn allow-scripts   @scarf/scarf@1.4.0 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.28.1 (install: (install scripts present))
npm warn allow-scripts   esbuild@0.25.12 (install: (install scripts present))
npm warn allow-scripts
npm warn allow-scripts Run `npm approve-scripts --allow-scripts-pending` to review, or `npm approve-scripts <pkg>` to allow.
✓ Frontend dependencies installed
✓ OpenAI API key loaded
✓ Google API key loaded


---

## 🎯 Chúng ta sẽ xây dựng gì?

Đầu tiên, bạn sẽ kết nối một ứng dụng MCP (Excalidraw) để agent có thể khởi chạy một bảng vẽ trắng (whiteboard) trực tiếp ngay bên trong khung chat:

> **Người dùng:** *Vẽ cho tôi một sơ đồ mạng đơn giản gồm 3 router, 2 laptop và 1 server bằng excalidraw*

<img src="images/mcp-apps-whiteboard.png" style="width: 50%; display: block; margin: 0 auto;">


Sau đó, bạn sẽ kích hoạt chế độ `openGenerativeUI` - cho phép agent tự động tạo ra UI tùy ý ngay trong thời gian thực:

> **Người dùng:** *Tạo hiệu ứng mưa taco!*

<img src="images/open-gen-ui.png" style="width: 50%; display: block; margin: 0 auto;">

---

## 📖 Khái niệm: GenUI mở là gì?

GenUI mở là pattern linh hoạt nhất trong toàn bộ hệ thống. Ở chế độ này, agent **không** bị giới hạn trong một tập hợp nhỏ các component đã đăng ký trước, cũng không bị bó buộc bởi một schema khai báo (dù là tĩnh hay động).

Sự linh hoạt này đến từ **MCP app**. Agent có thể tự động khám phá các công cụ ứng dụng từ một MCP server và mở các ứng dụng đó khi chúng phù hợp với yêu cầu của người dùng.

Trong kiến trúc này, frontend của bạn đóng vai trò là một **máy chủ lưu trữ (host)**. Nó không cần phải lập trình sẵn mọi giao diện UI có thể xảy ra; thay vào đó, nó chỉ cần kết nối môi trường chat với các ứng dụng bên ngoài tương thích.

### Tại sao nên sử dụng GenUI mở?

Cách tiếp cận này cực kỳ hữu ích khi các yêu cầu của người dùng quá rộng và không thể bao quát hết bằng một thư viện component cố định.

**✅ Ưu điểm:**
*   **Cực kỳ linh hoạt:** Agent có thể điều hướng người dùng vào các trải nghiệm ứng dụng phong phú, thay vì chỉ là các widget nhỏ nhúng trong dòng chat.
*   **Giảm thiểu sự phụ thuộc ở frontend:** Ứng dụng host của bạn có thể có thêm các tính năng mới một cách dễ dàng bằng cách kết nối với các MCP server.
*   **Phù hợp với các luồng công việc phức tạp:** Rất tuyệt vời cho các tác vụ như vẽ bảng trắng, thiết kế, lập kế hoạch, và các công việc mang tính chất sử dụng công cụ.

**❌ Nhược điểm:**
*   **Thiếu kiểm soát:** Bạn ít có quyền kiểm soát giao diện cuối cùng hơn so với phương pháp sử dụng component tĩnh hay schema.
*   **Phụ thuộc vào bên thứ 3:** Chất lượng đầu ra phụ thuộc vào các MCP app được kết nối và độ thông minh của agent trong việc chọn đúng ứng dụng.
*   **Bảo mật:** Đòi hỏi mức độ tin cậy, phân quyền và các rào chắn tích hợp chặt chẽ hơn.

### Đặc tả của MCP app

[MCP app](https://modelcontextprotocol.io/extensions/apps/overview) là một phần mở rộng của giao thức **Model Context Protocol**, cho phép các MCP server cung cấp giao diện tương tác tới các host hỗ trợ nó.

Kiến trúc này gồm 3 phần:
1. **Server (Máy chủ):** Cung cấp các công cụ và tài nguyên UI.
2. **Host (Nền tảng nhúng):** Nhúng UI vào một khung iframe được cô lập (sandboxed) và làm trung gian giao tiếp.
3. **View (Giao diện hiển thị):** Ứng dụng thực sự chạy bên trong iframe.

Một điểm thiết kế cốt lõi là **nâng cấp tuần tự**: Nếu host hỗ trợ MCP app, công cụ sẽ render ra UI phong phú. Nếu không, nó vẫn hoạt động như một công cụ MCP thông thường với đầu ra là văn bản thuần túy.

> 💡 **Lưu ý:** CopilotKit đã xử lý toàn bộ phần "hosting" - việc của bạn chỉ là trỏ nó tới URL của MCP server.

---

## 🚀 Thực hành 1: Khởi động server và tích hợp Excalidraw

Hãy khởi động backend và frontend của chúng ta. (Backend đã được cấu hình sẵn với LangGraph và Gemini trong file `server.py`).

In [2]:
# Khởi động backend (chạy ở port 8005)
from backend.server import start_backend
start_backend(port=8005)

# Khởi động frontend (chạy ở port 3005)
from helper import start_frontend
start_frontend(port=3005)

✓ Server running at http://localhost:8005
Starting frontend on port 3005 ...
✓ App running at http://localhost:3005

Read the logs: /home/cuong-ta/Documents/build-interactive-agents-with-generative-ui/4-open-generative-ui/frontend/dev-logs.txt


### Thêm MCP app server vào ứng dụng của bạn

CopilotKit hỗ trợ MCP app ngay từ đầu. Bạn chỉ cần truyền URL của máy chủ vào cấu hình `CopilotRuntime`. 

Chúng ta sẽ sử dụng [Excalidraw MCP app](https://github.com/excalidraw/excalidraw-mcp) làm ví dụ. Hãy ghi đè file `frontend/server.ts` bằng đoạn mã sau:

In [3]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const appAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8005",
});

const runtime = new CopilotRuntime({
  agents: { default: appAgent },
  // Cấu hình MCP Apps
  mcpApps: {
    servers: [
      {
        type: "http",
        url: "https://mcp.excalidraw.com", // URL của Excalidraw MCP server
        serverId: "example_mcp_server",
      },
    ],
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4005 }, () => {
  console.log("✓ CopilotKit API server running at http://localhost:4005");
});

Overwriting frontend/server.ts


**Những điểm cần lưu ý:**
*   `mcpApps.servers`: Kết nối runtime với một hoặc nhiều máy chủ MCP cung cấp công cụ app.
*   Mỗi mục cấu hình cho runtime biết nơi để khám phá các MCP app tương thích (ở đây là giao tiếp qua HTTP tới Excalidraw).
*   CopilotKit tự động cung cấp cho agent khả năng "khám phá" ứng dụng, để nó có thể gọi ra các công cụ này khi cần thiết trong quá trình chat.

### Trải nghiệm Excalidraw

Bây giờ agent của bạn đã có thể khởi chạy Excalidraw bên trong chat. Hãy mở giao diện người dùng và thử gửi đoạn prompt sau:

> *"Vẽ cho tôi một sơ đồ mạng đơn giản gồm 3 router, 2 laptop và 1 server bằng excalidraw"*

Bạn sẽ thấy agent tự động triệu hồi giao diện bảng vẽ và render sơ đồ!

> ⚠️ **Chú ý: GenUI mở là không thể đoán trước**
>
> Vì agent tạo ra UI từ con số không trong mỗi lần yêu cầu, kết quả có thể không hoàn toàn như bạn mong đợi ngay từ lần đầu tiên. Bạn có thể cần tinh chỉnh prompt (ví dụ: *"Thêm nhãn, tiêu đề và làm cho nội dung mạch lạc hơn"*) vài lần để biểu đồ trông đúng ý. 
> Đây là sự đánh đổi cốt lõi của GenUI: Bạn có được sự linh hoạt tối đa, nhưng phải hy sinh tính nhất quán. Đối với những UI cần độ ổn định 100%, hãy dùng phương pháp component có sẵn.

---

## 🚀 Thực hành 2: Kích hoạt `openGenerativeUI`

Với việc bật cờ `openGenerativeUI: true`, agent có thể tạo ra UI tùy ý - bao gồm HTML, CSS, JavaScript - và render trực tiếp vào trong chat. 

Hãy cập nhật lại file `frontend/server.ts` chỉ với 1 dòng thay đổi:

In [ ]:
%%writefile frontend/server.ts

import { serve } from "@hono/node-server";
import {
  CopilotRuntime,
  createCopilotEndpoint,
} from "@copilotkit/runtime/v2";
import { LangGraphHttpAgent } from "@copilotkit/runtime/langgraph";

const appAgent = new LangGraphHttpAgent({
  url: process.env.LANGGRAPH_DEPLOYMENT_URL || "http://localhost:8005",
});

const runtime = new CopilotRuntime({
  agents: { default: appAgent },
  openGenerativeUI: true, // Kích hoạt tính năng ở đây
  mcpApps: {
    servers: [
      {
        type: "http",
        url: "https://mcp.excalidraw.com",
        serverId: "example_mcp_server",
      },
    ],
  },
});

const app = createCopilotEndpoint({
  runtime,
  basePath: "/api/copilotkit",
});

serve({ fetch: app.fetch, port: 4005 }, () => {
  console.log("✓ CopilotKit API server running at http://localhost:4005");
});

Agent của bạn giờ đây đã sẵn sàng tạo ra các giao diện tùy ý từ sandbox. Hãy thử một yêu cầu sáng tạo:

> *"Tạo một cơn mưa taco!"*
> Hoặc:
> *"Tạo một thẻ với hiệu ứng mưa emoji taco"*

Agent sẽ viết mã HTML/CSS (dựa trên hướng dẫn từ system prompt trong backend) và trả về cho bạn một giao diện trực quan ngay trong khung chat mà không cần bạn phải cấu hình trước bất kỳ React component nào!

---

## 📚 Tổng kết những gì bạn đã học

*   **GenUI mở** loại bỏ ràng buộc phải đăng ký trước các components hay schema - agent có thể chạy các ứng dụng hoàn chỉnh hoặc tự viết mã render UI theo ý muốn.
*   **MCP app** cho phép agent mang đến các trải nghiệm ứng dụng phong phú (như Excalidraw) thông qua một giao thức tiêu chuẩn.
*   **`CopilotRuntime`** xử lý toàn bộ việc khám phá các ứng dụng MCP - bạn chỉ cần khai báo đường dẫn URL của server.
*   Cờ **`openGenerativeUI: true`** cấp quyền cho agent tạo và render trực tiếp HTML/CSS/JS ngay bên trong giao diện chat.

## 🔜 Bước tiếp theo

Ở **bài tiếp theo**, bạn sẽ vượt ra khỏi khuôn khổ GenUI cơ bản để xây dựng một ứng dụng quản lý công việc fullstack với **trạng thái chia sẻ** và **công cụ giao diện**. Agent và UI sẽ chia sẻ chung một dữ liệu thời gian thực - cả hai bên đều có thể đọc & ghi, và CopilotKit sẽ giữ cho chúng luôn được đồng bộ!